# GRPO fine tuning: teaching Qwen2.5 3B to cite its source

Reinforcement learning (GRPO) with four rule based rewards, so the model answers RAG questions in a strict
`<think>` → `<document id>` → `<answer>` format and cites the document it used. See the README for results.

> Runs on a free Colab T4. The dataset files (`filtered_train_stratified.jsonl`, `filtered_test.jsonl`) are not included.
> Logged output below comes from a run resumed at step 160 and stopped at step 400.


In [ ]:
#1
%%capture
# Install basic dependencies
!pip install datasets

import os
if "COLAB_" not in "".join(os.environ.keys()):
    # If not in Colab
    !pip install unsloth vllm
else:
    # In Colab
    !pip install --no-deps unsloth vllm

    # Clear problematic modules to prevent restart messages
    import sys, re, requests
    modules = list(sys.modules.keys())
    for x in modules:
        if "PIL" in x or "google" in x:
            sys.modules.pop(x)

    # Install additional packages needed
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft "trl==0.15.2" triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

    # vLLM extra requirements (skip transformers, numpy, xformers from their file)
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

# Git LFS install
!sudo apt-get install -y git-lfs
!git lfs install


In [ ]:
#2
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower
base_model_global = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_global,
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.6, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)

In [ ]:
#3
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
#4
from datasets import load_dataset, Dataset
import json
train_dataset = load_dataset("json", data_files="filtered_train_stratified.jsonl", split="train")
test_dataset  = load_dataset("json", data_files="filtered_test.jsonl",  split="train")  # json loader labels it "train"

def load_prompt(example):
    example["prompt"] = json.loads(example["prompt"])
    return example

train_dataset = train_dataset.map(load_prompt, num_proc=4)
test_dataset  = test_dataset.map(load_prompt,  num_proc=4)


In [ ]:
#5
import re

from difflib import SequenceMatcher

def document_id_accuracy_reward_func(completions, **kwargs):
    gold_ids = kwargs["document_id"]
    gold_docs = kwargs["documents"]
    rewards = []

    for completion, gold_id, docs in zip(completions, gold_ids, gold_docs):
        if isinstance(completion, list):
            completion = completion[0]
        content = completion["content"]

        # Extract <document id>
        docid_match = re.search(r"<document\s*id>\s*(\d+)\s*</document\s*id>", content, re.IGNORECASE)
        predicted_id = int(docid_match.group(1)) if docid_match else None

        # Extract <answer> block
        answer_match = re.search(r"<answer>(.*?)</answer>", content, re.DOTALL | re.IGNORECASE)
        answer_content = answer_match.group(1).strip().lower() if answer_match else ""

        reward = 0.0

        # 1. Exact doc ID match
        if predicted_id == gold_id:
            reward += 0.65

        # 2. Fuzzy string overlap between answer and gold doc
        if isinstance(docs, list) and 0 <= gold_id < len(docs) and answer_content:
            gold_doc = docs[gold_id].strip().lower()
            overlap_ratio = SequenceMatcher(None, gold_doc, answer_content).ratio()
            if overlap_ratio > 0.3:
                reward += 0.35

        rewards.append(min(reward, 1.0))

    return rewards





def tag_existence_reward_func(completions, **kwargs):
    rewards = []
    for completion in completions:
        if isinstance(completion, list):
            completion = completion[0]
        content = completion["content"]
        score = 0.0
        if "<think>" in content and "</think>" in content:
            score += 0.3
        if "<document id>" in content and "</document id>" in content:
            score += 0.3
        if "<answer>" in content and "</answer>" in content:
            score += 0.4
        rewards.append(score)
    return rewards



def no_extra_text_reward_func(completions, **kwargs):
    pattern = r"^\s*<think>.*?</think>\s*<document id>\s*\d+\s*</document id>\s*<answer>.*?</answer>\s*$"
    rewards = []
    for completion in completions:
        if isinstance(completion, list):
            completion = completion[0]
        content = completion["content"]
        rewards.append(1.0 if re.fullmatch(pattern, content, re.DOTALL) else 0.0)
    return rewards



def tag_order_reward_func(completions, **kwargs):
    rewards = []
    for completion in completions:
        if isinstance(completion, list):
            completion = completion[0]
        content = completion["content"]
        think_pos = content.find("<think>")
        doc_pos = content.find("<document id>")
        answer_pos = content.find("<answer>")
        if -1 in [think_pos, doc_pos, answer_pos]:
            rewards.append(0.0)
        elif think_pos < doc_pos < answer_pos:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards


In [ ]:
#6
from huggingface_hub import notebook_login
from huggingface_hub import HfApi, create_repo
notebook_login()

In [ ]:
#7
repo_name = f"genie-grpo-test-RAG-Qwen3B-faraya"

username = HfApi().whoami()["name"]
full_repo_name = f"{username}/{repo_name}"
create_repo(repo_id=repo_name, exist_ok=True)

In [ ]:
#8
from trl import GRPOConfig, GRPOTrainer
from huggingface_hub import HfFolder
from transformers import TrainerCallback
from transformers import PreTrainedModel, PreTrainedTokenizerBase
import shutil
import os
max_prompt_length = 768
hf_token = os.getenv("HF_TOKEN", HfFolder.get_token())


from huggingface_hub import upload_folder

class EvalCheckpointPushCallback(TrainerCallback):
    def __init__(self, hub_model_id):
        self.hub_model_id = hub_model_id

    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        if step % 10 != 0:
            return  # ❌ Only push every 10th step

        ckpt_src = os.path.join(args.output_dir, f"checkpoint-{step}")
        ckpt_temp = os.path.join(args.output_dir, f"hf-format-step-{step}")

        print(f"🔔 on_save called at step {step}")
        print(f"📁 Checking for: {ckpt_src}")

        if os.path.exists(ckpt_src):
            os.makedirs(ckpt_temp, exist_ok=True)
            for fname in os.listdir(ckpt_src):
                shutil.copy2(os.path.join(ckpt_src, fname), os.path.join(ckpt_temp, fname))

            with open(os.path.join(ckpt_temp, "README.md"), "w") as f:
                f.write(f"# Checkpoint at step {step}\n\nThis checkpoint was saved during training step {step}.")

            print(f"🚀 Uploading {ckpt_temp} to HF repo {self.hub_model_id}/step-{step}")
            upload_folder(
                repo_id=self.hub_model_id,
                folder_path=ckpt_temp,
                path_in_repo=f"step-{step}",
                commit_message=f"Add checkpoint step-{step}",
                repo_type="model"
            )
        else:
            print(f"⚠️ Checkpoint folder missing: {ckpt_src} — not pushed")




training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "paged_adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 2,
    num_generations = 6,
    max_prompt_length = 768,
    max_completion_length = 256,
    max_steps = 500,
    # num_train_epochs = 1, # Set to 1 for a full training run
    save_steps = 10,                    # Save every 10 steps
    save_total_limit = None,              # Keep only the latest checkpoint
    push_to_hub = True,                # Enable pushing to HF Hub
    hub_strategy = "checkpoint",       # Push every time we save
    hub_model_id = full_repo_name,
    hub_token = hf_token,
    load_best_model_at_end = False,    # Not needed for GRPO (optional)
    output_dir = "outputs_for_rag_qwen",            # Directory for checkpoints
    report_to = "none",                # Logging service
    disable_tqdm = False,        # ✅ enables stdout progress + logging
    log_level = "info",          # ✅ print logs at every step
)

In [ ]:
#9 - Fetch last saved ckpt from HF to resume training
from huggingface_hub import snapshot_download
import os

# Download the specific checkpoint folder from HF Hub
checkpoint_dir = snapshot_download(
    repo_id="faraya1/genie-grpo-test-RAG-Qwen3B-faraya",
    repo_type="model",
    allow_patterns=["step-160/*"],
    local_dir="outputs_for_rag_qwen",  # This matches your training_args.output_dir
    local_dir_use_symlinks=False,
)

# Define full local path to the checkpoint folder
resume_checkpoint_path = os.path.join(checkpoint_dir, "step-160")


In [ ]:
#9
reward_funcs = [
    document_id_accuracy_reward_func,
    tag_existence_reward_func,
    no_extra_text_reward_func,
    tag_order_reward_func
]

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_funcs,
    args=training_args,
    train_dataset=train_dataset,
    callbacks=[EvalCheckpointPushCallback(hub_model_id=full_repo_name)],
)
trainer.train(resume_from_checkpoint=resume_checkpoint_path)

Step,Training Loss,reward,reward_std,completion_length,kl,rewards / document_id_accuracy_reward_func,rewards / tag_existence_reward_func,rewards / no_extra_text_reward_func,rewards / tag_order_reward_func
161,0.006800,3.629167,0.379225,106.083336,0.168868,0.795833,1.000000,0.833333,1.000000
162,0.002900,3.650000,0.000000,80.166668,0.011766,0.650000,1.000000,1.000000,1.000000
163,0.001400,3.541667,0.167829,128.083336,0.004604,0.541667,1.000000,1.000000,1.000000
164,0.001400,3.533334,0.285774,128.666668,0.004701,0.650000,0.966667,0.916667,1.000000
165,0.001200,3.716667,0.167829,95.250000,0.030766,0.716667,1.000000,1.000000,1.000000
166,0.000300,3.525000,0.371953,119.000000,0.006291,0.525000,1.000000,1.000000,1.000000
167,0.001600,3.595834,0.132681,98.583336,0.010426,0.595833,1.000000,1.000000,1.000000
168,0.002000,3.433334,0.167829,112.416668,0.020665,0.433333,1.000000,1.000000,1.000000
169,0.001500,3.708334,0.090370,112.083336,0.006560,0.708333,1.000000,1.000000,1.000000
170,0.001000,3.545834,0.268379,94.000004,0.025292,0.545833,1.000000,1.000000,1.000000
